In [0]:
#Criando o schema da camada Bronze para organizar as tabelas brutas
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:
#Lendo o CSV com o cabeçalho e mantenho as colunas sem inferência automática de tipos. Mantenho os campos como texto nesta etapa para preservar o conteúdo original e tipar os dados só na Silver
df_info_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/cinedata_inputs/movies_info_TMDB_IMDB.csv"))

#Conferindo algumas linhas para validar a leitura
display(df_info_raw.limit(10))

In [0]:
from pyspark.sql import functions as F

#Registrando quando o arquivo foi carregado na camada Bronze
df_info_raw = df_info_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

display(df_info_raw.limit(10))

In [0]:
#Salvando os dados como Delta na tabela Bronze
df_info_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_info_filmes")

#Conferindo se a tabela foi salva
display(spark.table("workspace.bronze.tb_info_filmes").limit(10))

In [0]:
#Conferindo que a tabela existe
spark.sql("SHOW TABLES IN workspace.bronze").display()

In [0]:
#Lendo o arquivo financeiro mantendo os dados como vieram no CSV. Nesta célula leio o CSV sem limpar os valores, para manter uma cópia fiel da origem
df_financeiro_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/cinedata_inputs/movies_financials_IMDB_TMDB.csv"))

display(df_financeiro_raw.limit(10))

In [0]:
#Registrando quando os dados foram carregados na Bronze
df_financeiro_raw = df_financeiro_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

#Salvando os dados como Delta e conferindo as primeiras linhas
df_financeiro_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_financeiro_filmes")

display(spark.table("workspace.bronze.tb_financeiro_filmes").limit(10))

In [0]:
#Lendo o arquivo de métricas mantendo os valores como vieram no CSV
df_metricas_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/cinedata_inputs/movies_metrics_IMDB_TMDB.csv"))

display(df_metricas_raw.limit(10))

In [0]:
#Registrando quando os dados foram carregados na Bronze
df_metricas_raw = df_metricas_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

#Salvando os dados como Delta e conferindo algumas linhas
df_metricas_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_metricas_engajamento")

display(spark.table("workspace.bronze.tb_metricas_engajamento").limit(10))

In [0]:
#Lendo o arquivo com informações de gêneros, elenco e produção
df_creditos_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/cinedata_inputs/credits_and_tags_IMDB_TMDB.csv"))

display(df_creditos_raw.limit(10))

In [0]:
#Registrando quando os dados foram carregados na Bronze
df_creditos_raw = df_creditos_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

#Salvando os dados como Delta e conferindo algumas linhas
df_creditos_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_creditos_tags")

display(spark.table("workspace.bronze.tb_creditos_tags").limit(10))

In [0]:
#Lendo o arquivo de avaliações mantendo os valores como vieram no CSV
df_avaliacoes_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/cinedata_inputs/movies_reviews.csv"))

display(df_avaliacoes_raw.limit(10))

In [0]:
#Registrando quando os dados foram carregados na Bronze
df_avaliacoes_raw = df_avaliacoes_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

#Salvando os dados como Delta e conferindo algumas linhas
df_avaliacoes_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_avaliacoes_usuarios")

display(spark.table("workspace.bronze.tb_avaliacoes_usuarios").limit(10))

In [0]:
#Verificando se as 5 tabelas estão no sistema Bronze
spark.sql("SHOW TABLES IN workspace.bronze").display()

In [0]:
#Selecionando o período que será consultado na API do Banco Central
dbutils.widgets.text("data_inicial", "10-01-2026", "Data inicial (MM-DD-YYYY)")
dbutils.widgets.text("data_final", "10-07-2026", "Data final (MM-DD-YYYY)")

print(dbutils.widgets.get("data_inicial"))
print(dbutils.widgets.get("data_final"))

In [0]:
import requests

#Pegando as datas preenchidas nos widgets
data_inicial = dbutils.widgets.get("data_inicial")
data_final = dbutils.widgets.get("data_final")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)")

parametros = {
    "@dataInicial": f"'{data_inicial}'",
    "@dataFinalCotacao": f"'{data_final}'",
    "$format": "json"}

resposta = requests.get(url, params=parametros)
resposta.raise_for_status()

cotacoes = resposta.json()["value"]

#Transformando o retorno da API em um DataFrame para conferir os dados
df_cotacao_raw = spark.createDataFrame(cotacoes)

display(df_cotacao_raw.limit(10))

In [0]:
#Registrando quando as cotações foram consultadas
df_cotacao_raw = df_cotacao_raw.withColumn(
    "ingestion_datetime",
    F.current_timestamp())

#Salvando o retorno da API como Delta na Bronze
df_cotacao_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_cotacao_dolar")

display(spark.table("workspace.bronze.tb_cotacao_dolar").limit(10))